# **ST 554 Spring 2026 Homework Ten**
## *Created by Cody Ashby on April 20, 2026*

This assignment will allow us to use Spark structured streaming so we can work with streaming data.

### Part I - Creating Streaming Data Using the `rate` Format

To begin with, we'll create some streaming data and perform a few elementary functions on a collection of values. As always, let's import the appropriate modules and components we'll need.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.types import StructType
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.sql.functions import col, sqrt, pmod

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:21:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:21:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/21 16:21:19 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Here, we'll use the `rate` format to make our own streaming data; namely, we'll find the square root for each value as well as it's remainder when divided by four. Once our query begins, this stream will be written to the `memory` so we won't see results right away until we officially stop the query. We'll see the results after roughly 30 seconds.

In [2]:
rate_df = spark.readStream.format("rate").load()
rootmod_df = rate_df.withColumns({"square_root": sqrt(col("value")),
                                     "mod_four": pmod(col("value"),4)})
collection_df = rootmod_df.writeStream.format("memory").queryName("Operations").start()

26/04/21 16:21:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9085e995-b328-4c99-bf06-ea964bb53658. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:21:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


It looks like enough time has elapsed, so we can stop the query and see out results.

In [3]:
collection_df.stop()
spark.sql("select * from Operations").show()

26/04/21 16:22:03 WARN DAGScheduler: Failed to cancel job group b001c114-6a4a-4ebe-b7ec-1701ad5334c8. Cannot find active jobs for it.
26/04/21 16:22:03 WARN DAGScheduler: Failed to cancel job group b001c114-6a4a-4ebe-b7ec-1701ad5334c8. Cannot find active jobs for it.


+--------------------+-----+------------------+--------+
|           timestamp|value|       square_root|mod_four|
+--------------------+-----+------------------+--------+
|2026-04-21 16:21:...|    0|               0.0|       0|
|2026-04-21 16:21:...|    1|               1.0|       1|
|2026-04-21 16:21:...|    2|1.4142135623730951|       2|
|2026-04-21 16:21:...|    3|1.7320508075688772|       3|
|2026-04-21 16:21:...|    4|               2.0|       0|
|2026-04-21 16:21:...|    5|  2.23606797749979|       1|
|2026-04-21 16:21:...|    6| 2.449489742783178|       2|
|2026-04-21 16:21:...|    7|2.6457513110645907|       3|
|2026-04-21 16:21:...|    8|2.8284271247461903|       0|
|2026-04-21 16:21:...|    9|               3.0|       1|
|2026-04-21 16:21:...|   10|3.1622776601683795|       2|
|2026-04-21 16:21:...|   11|   3.3166247903554|       3|
|2026-04-21 16:21:...|   12|3.4641016151377544|       0|
|2026-04-21 16:21:...|   13| 3.605551275463989|       1|
|2026-04-21 16:21:...|   14|3.7

That seemed pretty simple after getting familiar with the vernacular associated with this idea of data streams.

### Part II - Using Data From a CSV with a Pipeline

Next, we'll use a real dataset (based on the `bike` data set that's been used to demonsrate modeling techniques) and combine what we've done so far by starting with reading in a csv file to create a Spark SQL-style dataframe.

In [4]:
bike_data = spark.read.csv("bikeDetails_for_fit.csv",inferSchema=True,header=True)

Second, we'll use the `SQLTransformer` to write out a SQL statement that chooses a few columns. A couple of log transformations and the creation of a dummy variable will also be applied.

In [5]:
sql_statement = SQLTransformer(
    statement="""SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
                    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
                    FROM __THIS__""")

The `VectorAssembler` will now be used to create a `features` column so that we can fit a model if need be.

In [6]:
assembling_vectors = VectorAssembler(inputCols=['year','log_km_driven','one_owner'],outputCol='features',handleInvalid='keep')

A pipeline will now be created from these two actions we've just done. For the time being, we'll hold off on what type of estimator to use (regression vs. classification).

In [7]:
pipeline = Pipeline(stages=[sql_statement,assembling_vectors])

With this pipeline, we'll do some fitting with the dataset that was read in not too long ago.

In [8]:
fitted_pipeline = pipeline.fit(bike_data)

This fitted pipeline will now be used to construct a reading stream that allows us to search for extra csv files in a certain location (i.e. a separate file folder).

In [9]:
bike_stream = spark.readStream.schema(bike_data.schema).csv("Extra Bike Files",inferSchema=True,header=True)

Every time a new csv file comes into the folder, we'll use the `transform()` method to read it in to the pipeline defined earlier.

In [10]:
transformed_bike_pipeline = fitted_pipeline.transform(bike_stream)

Lastly, we'll create a writing stream that will display the output of each file 

In [11]:
new_bike_stream = transformed_bike_pipeline.writeStream.outputMode("append").format("console").start()

26/04/21 16:23:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ffc6636a-8176-4b9f-99ec-986dbbae7029. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:23:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [12]:
#After inserting all of the files into the folder, we need to stop the query.
new_bike_stream.stop()

26/04/21 16:24:20 WARN DAGScheduler: Failed to cancel job group 626f92fc-a4d5-4525-8d2d-9db41f6213fb. Cannot find active jobs for it.
26/04/21 16:24:20 WARN DAGScheduler: Failed to cancel job group 626f92fc-a4d5-4525-8d2d-9db41f6213fb. Cannot find active jobs for it.


And there you have it! A nice and quick demonstration of using data streams.